# Seminar 10: Flow Matching

**Deep Learning Course 2026**

**Author:** Nikita Kiselev

In [ ]:
import torch
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["font.size"] = 16

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

## 1. Flow Matching in 2D

We start with a toy 2D distribution, a spiral, so that we can see every sample and every trajectory.

In [ ]:
def create_spiral_data(num_samples=100_000):
    theta = 6 * torch.pi * torch.rand(num_samples)
    r = theta / (2 * torch.pi)
    x = r * torch.cos(theta) + 0.1 * torch.randn(num_samples)
    y = r * torch.sin(theta) + 0.1 * torch.randn(num_samples)
    return torch.stack([x, y], dim=1)

In [ ]:
train_dataset = create_spiral_data()
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [ ]:
plt.scatter(*train_dataset[:2000].T, s=2)
plt.axis("equal")
plt.show()

**Flow Matching objective**

We move samples from the noise $\mathbf{x}_0 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ to the data $\mathbf{x}_1 \sim \pi(\mathbf{x})$ along straight lines:
$$
\mathbf{x}_t = t \mathbf{x}_1 + (1 - t) \mathbf{x}_0, \qquad \frac{d\mathbf{x}_t}{dt} = \mathbf{x}_1 - \mathbf{x}_0.
$$

A network $\mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}_t, t)$ learns this velocity by regression:
$$
\mathbb{E}_{t \sim \mathcal{U}[0, 1]}\, \mathbb{E}_{\mathbf{x}_1 \sim \pi(\mathbf{x})}\, \mathbb{E}_{\mathbf{x}_0 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})} \left\| (\mathbf{x}_1 - \mathbf{x}_0) - \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) \right\|_2^2 \to \min_{\boldsymbol{\theta}}
$$

Note the convention: here $\mathbf{x}_0$ is noise and $\mathbf{x}_1$ is data, the opposite of DDPM.

**Building the model**

The model takes two inputs: the point $\mathbf{x}_t$ and the time $t$.

We map both into a common hidden space and add them.

To embed a scalar $t$, we use sinusoidal features $\cos(k \pi t), \sin(k \pi t)$ of different frequencies $k$, as in Transformers.

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.register_buffer("freqs", torch.arange(1, dim // 2 + 1) * torch.pi)

    def forward(self, t):
        emb = self.freqs * t  # [b, 1] * [dim / 2] -> [b, dim / 2]
        return torch.cat([emb.cos(), emb.sin()], dim=-1)

In [ ]:
class FlowMatchingModel(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=512):
        super().__init__()
        self.x_proj = nn.Linear(input_dim, hidden_dim)
        self.t_proj = TimeEmbedding(hidden_dim)
        self.net = nn.Sequential(
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x, t):
        return self.net(self.x_proj(x) + self.t_proj(t))

**Training the model**

Every training step follows the objective:
1. Sample data $\mathbf{x}_1 \sim \pi(\mathbf{x})$, noise $\mathbf{x}_0 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and time $t \sim \mathcal{U}[0, 1]$
2. Get the point on the line $\mathbf{x}_t = t \mathbf{x}_1 + (1 - t) \mathbf{x}_0$
3. Regress the velocity: $\mathcal{L} = \left\| (\mathbf{x}_1 - \mathbf{x}_0) - \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) \right\|_2^2$

In [ ]:
epochs = 100
learning_rate = 1e-3

model = FlowMatchingModel().to(device)
optimizer = optim.AdamW(model.parameters(), learning_rate)

model.train()

for epoch in range(epochs):
    total = 0
    for x_1 in train_loader:
        x_1 = x_1.to(device)
        x_0 = torch.randn_like(x_1)
        t = torch.rand(x_1.shape[0], 1, device=device)
        x_t = t * x_1 + (1 - t) * x_0
        loss = F.mse_loss(model(x_t, t), x_1 - x_0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * x_1.shape[0]
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: loss {total / len(train_dataset):.3f}")

**Sampling with the Euler method**

To generate data, we solve the ODE $\frac{d\mathbf{x}}{dt} = \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}, t)$ from $t = 0$ to $t = 1$ with $N$ Euler steps of size $h = 1 / N$:
$$
\mathbf{x}(t + h) = \mathbf{x}(t) + h \cdot \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}(t), t)
$$

We keep every intermediate point to look at the trajectories later.

In [ ]:
@torch.no_grad()
def sample(model, x, num_steps=100):
    model.eval()
    trajectory = [x]
    for t in torch.linspace(0, 1, num_steps + 1)[:-1]:
        t = torch.full((x.shape[0], 1), t.item(), device=device)
        x = x + model(x, t) / num_steps
        trajectory.append(x)
    return torch.stack(trajectory).cpu()  # [num_steps + 1, b, 2]

In [ ]:
trajectory = sample(model, torch.randn(2000, 2, device=device))

plt.scatter(*trajectory[-1].T, s=2)
plt.axis("equal")
plt.show()

**Trajectories**

Every training pair $(\mathbf{x}_0, \mathbf{x}_1)$ is connected by a straight line.

Are the learned trajectories straight too? Why?

In [ ]:
plt.plot(trajectory[:, :100, 0], trajectory[:, :100, 1], color="gray", linewidth=0.5)
plt.scatter(*trajectory[0, :100].T, s=5, label="noise")
plt.scatter(*trajectory[-1, :100].T, s=5, label="data")
plt.legend()
plt.axis("equal")
plt.show()

**How many steps do we need?**

Sampling costs one network call per step, so we want as few steps as possible.

Let's start from the same noise and vary the number of Euler steps. What does a single step produce, and why?

In [ ]:
x_0 = torch.randn(2000, 2, device=device)

plt.figure(figsize=(20, 4))
for i, num_steps in enumerate([1, 2, 5, 20, 100]):
    plt.subplot(1, 5, i + 1)
    plt.scatter(*sample(model, x_0, num_steps)[-1].T, s=2)
    plt.axis([-3.5, 3.5, -3.5, 3.5])
    plt.title(f"{num_steps} steps")
plt.show()

## 2. Conditional Flow Matching on MNIST

We return to MNIST from the previous seminar, now in grayscale.

We scale pixels to $[-1, 1]$, so the data and the noise have a similar scale.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: 2 * x - 1
])
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True, num_workers=4)

**Conditioning on a class**

To generate a given digit $y$, we add one more embedding to the same model: $\mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}_t, t, y)$.

We also reserve an extra class $\varnothing$ = "no label" (index 10) for classifier-free guidance.

In [ ]:
class ConditionalFlowMatchingModel(FlowMatchingModel):
    def __init__(self, input_dim=784, hidden_dim=2048, num_classes=10):
        super().__init__(input_dim, hidden_dim)
        self.y_proj = nn.Embedding(num_classes + 1, hidden_dim)  # NOTE: the last class is "no label"

    def forward(self, x, t, y):
        return self.net(self.x_proj(x) + self.t_proj(t) + self.y_proj(y))

**Training with label dropout**

The training loop is the same, but with probability $0.1$ we replace the label with $\varnothing$.

So a single model learns both the conditional velocity $\mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}_t, t, y)$ and the unconditional one $\mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}_t, t, \varnothing)$.

In [ ]:
epochs = 40
learning_rate = 3e-4

model = ConditionalFlowMatchingModel().to(device)
optimizer = optim.AdamW(model.parameters(), learning_rate)

model.train()

for epoch in tqdm(range(epochs), desc="Epoch"):
    total = 0
    for x_1, y in train_loader:
        x_1, y = x_1.view(-1, 784).to(device), y.to(device)
        y = torch.where(torch.rand(y.shape[0], device=device) < 0.1, 10, y)  # NOTE: label dropout
        x_0 = torch.randn_like(x_1)
        t = torch.rand(x_1.shape[0], 1, device=device)
        x_t = t * x_1 + (1 - t) * x_0
        loss = F.mse_loss(model(x_t, t, y), x_1 - x_0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * x_1.shape[0]
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: loss {total / len(train_dataset):.3f}")

**Classifier-free guidance**

At every Euler step we mix the two velocities with a guidance scale $w$:
$$
\mathbf{f}_w = \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}, t, \varnothing) + w \cdot \left( \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}, t, y) - \mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x}, t, \varnothing) \right)
$$

$w = 0$ ignores the label, $w = 1$ is the plain conditional model, and $w > 1$ pushes samples further towards the class.

In [ ]:
@torch.no_grad()
def sample_cfg(model, y, w, num_steps=50):
    model.eval()
    x = torch.randn(y.shape[0], 784, device=device)
    y_null = torch.full_like(y, 10)
    for t in torch.linspace(0, 1, num_steps + 1)[:-1]:
        t = torch.full((y.shape[0], 1), t.item(), device=device)
        f_cond, f_uncond = model(x, t, y), model(x, t, y_null)
        x = x + (f_uncond + w * (f_cond - f_uncond)) / num_steps
    return x.view(-1, 1, 28, 28).clamp(-1, 1).cpu()

Each row below uses its own guidance scale $w = 0, 1, 3, 5$; each column asks for a digit from 0 to 9.

In [ ]:
y = torch.arange(10, device=device)
images = torch.cat([sample_cfg(model, y, w) for w in [0, 1, 3, 5]])

plt.figure(figsize=(12, 5))
plt.imshow(make_grid(images, nrow=10, normalize=True).permute(1, 2, 0))
plt.axis("off")
plt.show()

What do we gain and what do we lose as $w$ grows? Try a larger $w$, e.g. $w = 10$.